# Ask Andela — RAG Pipeline

**Version:** 1.0 | **Date:** March 2026 | **Team:** Eben, Amit, Phillip, Kayode, Tunde, Mugao

This notebook implements the full RAG (Retrieval-Augmented Generation) pipeline for Ask Andela — an AI study assistant grounded in the AI Engineering Bootcamp's own chats, resources and course materials.

---

### How this notebook is organised (for splitting into separate files later)

| Part | Description | Future file |
|------|-------------|-------------|
| 1 | Document Loading | `01_load_documents.py` |
| 2 | Text Chunking | `02_chunk_documents.py` |
| 3 | Embeddings & ChromaDB Vector Store | `03_build_vectorstore.py` |
| 4 | Retrieval | `04_retrieval.py` |
| 5 | LLM via OpenRouter | `05_llm.py` |
| 6 | Full RAG Pipeline | `06_rag_pipeline.py` |
| 7 | Gradio UI | `07_gradio_ui.py` |
| 8 | Evaluation Framework | `08_evaluation.py` |

---

### Architecture

```
User Query (Gradio)
       │
       ▼
  Query Embedding (all-MiniLM-L6-v2)
       │
       ▼
ChromaDB Vector Store ──→ Top-K Retrieved Chunks (with source metadata)
       │
       ▼
 Prompt Assembly [System Prompt + Context + Query]
       │
       ▼
OpenRouter LLM (meta-llama/llama-3.1-8b-instruct:free)
       │
       ▼
Answer + Source Citation (Gradio output)
```

## Setup

Install dependencies if not already installed. All packages are in `requirements.txt`.

In [ ]:
# Uncomment to install if running in a fresh environment
# !pip install chromadb sentence-transformers openai python-dotenv tiktoken gradio

In [1]:
import os
import json
import time
from pathlib import Path
from typing import Optional

import tiktoken
import chromadb
import gradio as gr
from dotenv import load_dotenv
from openai import OpenAI
from sentence_transformers import SentenceTransformer

print("All imports successful.")

All imports successful.


In [14]:
# ── Configuration ────────────────────────────────────────────────────────────
# Paths (relative to this notebook in scripts/)
NOTEBOOK_DIR = Path(".").resolve()
PROJECT_ROOT = NOTEBOOK_DIR.parent
DATA_DIR     = PROJECT_ROOT / "data"
CHROMA_DIR   = PROJECT_ROOT / "chroma_db"

# Embedding model (free, fast, runs locally via HuggingFace)
EMBEDDING_MODEL = "sentence-transformers/all-MiniLM-L6-v2"

# OpenRouter model — change to any model available on openrouter.ai
# Free options:  "meta-llama/llama-3.1-8b-instruct"
#                "qwen/qwen-2.5-7b-instruct"
#                "mistralai/mistral-7b-instruct"
# Paid (better): "openai/gpt-4o-mini", "anthropic/claude-3-haiku"
LLM_MODEL = "meta-llama/llama-3.1-8b-instruct"

# Chunking strategy (per PRD spec)
CHUNK_SIZE    = 400   # tokens
CHUNK_OVERLAP = 50    # tokens

# Retrieval
TOP_K = 5

# Load API keys from .env
load_dotenv(PROJECT_ROOT / ".env")
OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY")

if not OPENROUTER_API_KEY:
    raise ValueError("OPENROUTER_API_KEY not found. Check your .env file.")

print(f"Project root : {PROJECT_ROOT}")
print(f"Data dir     : {DATA_DIR}")
print(f"ChromaDB dir : {CHROMA_DIR}")
print(f"LLM model    : {LLM_MODEL}")
print(f"OpenRouter key loaded: {'✓' if OPENROUTER_API_KEY else '✗'}")

Project root : /Users/severus/Documents/ai_bootcamp/ask_andela_cohort
Data dir     : /Users/severus/Documents/ai_bootcamp/ask_andela_cohort/data
ChromaDB dir : /Users/severus/Documents/ai_bootcamp/ask_andela_cohort/chroma_db
LLM model    : meta-llama/llama-3.1-8b-instruct
OpenRouter key loaded: ✓


---
## Part 1 — Document Loading
**Future file:** `01_load_documents.py`

Load all `.txt` files from the `data/` folder. Assign category metadata based on filename prefix:
- `channel_*` → discourse channel exports
- `resource_*` → course/programme resources

In [3]:
def get_category(filename: str) -> str:
    """Infer document category from filename prefix."""
    stem = Path(filename).stem
    if stem.startswith("channel_"):
        return "discourse_channel"
    elif stem.startswith("resource_"):
        return "course_resource"
    elif stem.startswith("week"):
        return "week_content"
    return "other"


def load_documents(data_dir: Path) -> list[dict]:
    """
    Load all .txt (and .pdf via PyMuPDF if available) files from data_dir.

    Returns a list of dicts:
        {
            "content": str,
            "metadata": {"source": filename, "category": category}
        }
    """
    documents = []

    # Load plain-text files
    for file_path in sorted(data_dir.glob("*.txt")):
        try:
            content = file_path.read_text(encoding="utf-8").strip()
            if not content:
                print(f"  [skip] {file_path.name} — empty file")
                continue
            documents.append({
                "content": content,
                "metadata": {
                    "source":   file_path.name,
                    "category": get_category(file_path.name),
                }
            })
            print("documents length..... ", len(documents))
        except Exception as e:
            print(f"  [error] {file_path.name}: {e}")

    return documents

In [4]:
documents = load_documents(DATA_DIR)

print(f"Loaded {len(documents)} documents\n")
print(f"{'Filename':<50} {'Category':<20} {'Chars':>8}")
print("-" * 82)
for doc in documents:
    print(f"{doc['metadata']['source']:<50} {doc['metadata']['category']:<20} {len(doc['content']):>8,}")

total_chars = sum(len(d["content"]) for d in documents)
print("-" * 82)
print(f"Total characters: {total_chars:,}")

documents length.....  1
documents length.....  2
documents length.....  3
documents length.....  4
documents length.....  5
documents length.....  6
documents length.....  7
documents length.....  8
documents length.....  9
documents length.....  10
documents length.....  11
documents length.....  12
documents length.....  13
Loaded 13 documents

Filename                                           Category                Chars
----------------------------------------------------------------------------------
channel_general_chat.txt                           discourse_channel       3,205
channel_igniters_information.txt                   discourse_channel       6,288
channel_protocol_daily_updates.txt                 discourse_channel       1,528
channel_solidroad_behavioral_assignments.txt       discourse_channel       6,491
channel_staff_announcements.txt                    discourse_channel       8,517
resource_additional_learning_material.txt          course_resource         8,961


---
## Part 2 — Text Chunking
**Future file:** `02_chunk_documents.py`

Split documents into overlapping token-based chunks as specified in the PRD:
- `chunk_size = 400` tokens
- `chunk_overlap = 50` tokens

Using `tiktoken` (`cl100k_base` encoding — compatible with most modern LLMs) for accurate token counting.
Metadata preserved per chunk: `source`, `category`, `chunk_index`, `total_chunks`.

In [5]:
def chunk_text(
    text: str,
    chunk_size: int = CHUNK_SIZE,
    chunk_overlap: int = CHUNK_OVERLAP,
    encoding_name: str = "cl100k_base",
) -> list[str]:
    """
    Split a string into overlapping token-based chunks.

    The final chunk may be smaller than chunk_size.
    Chunks are split on whitespace boundaries where possible to avoid
    cutting words mid-token.
    """
    enc = tiktoken.get_encoding(encoding_name)
    tokens = enc.encode(text)

    chunks = []
    start = 0
    while start < len(tokens):
        end = min(start + chunk_size, len(tokens))
        chunk_tokens = tokens[start:end]
        chunks.append(enc.decode(chunk_tokens))
        if end == len(tokens):
            break
        start += chunk_size - chunk_overlap

    return chunks


def chunk_documents(
    documents: list[dict],
    chunk_size: int = CHUNK_SIZE,
    chunk_overlap: int = CHUNK_OVERLAP,
) -> list[dict]:
    """
    Chunk all documents and propagate metadata to every chunk.

    Returns a flat list of chunk dicts:
        {
            "content":  str,
            "metadata": {"source", "category", "chunk_index", "total_chunks"}
        }
    """
    all_chunks = []
    for doc in documents:
        text_chunks = chunk_text(doc["content"], chunk_size, chunk_overlap)
        for i, chunk in enumerate(text_chunks):
            all_chunks.append({
                "content": chunk,
                "metadata": {
                    **doc["metadata"],
                    "chunk_index":  i,
                    "total_chunks": len(text_chunks),
                }
            })
    return all_chunks

In [6]:
all_chunks = chunk_documents(documents, CHUNK_SIZE, CHUNK_OVERLAP)

enc = tiktoken.get_encoding("cl100k_base")
chunk_token_counts = [len(enc.encode(c["content"])) for c in all_chunks]

print(f"Total chunks   : {len(all_chunks)}")
print(f"Avg tokens/chunk : {sum(chunk_token_counts) / len(chunk_token_counts):.0f}")
print(f"Min tokens/chunk : {min(chunk_token_counts)}")
print(f"Max tokens/chunk : {max(chunk_token_counts)}")
print()

# Chunks per document
from collections import Counter
source_counts = Counter(c["metadata"]["source"] for c in all_chunks)
print(f"{'Source':<50} {'Chunks':>6}")
print("-" * 58)
for source, count in sorted(source_counts.items()):
    print(f"{source:<50} {count:>6}")

# Preview first chunk
print("\n--- First chunk preview ---")
print(f"Source  : {all_chunks[0]['metadata']['source']}")
print(f"Category: {all_chunks[0]['metadata']['category']}")
print(f"Content : {all_chunks[0]['content'][:300]}...")

Total chunks   : 44
Avg tokens/chunk : 352
Min tokens/chunk : 59
Max tokens/chunk : 400

Source                                             Chunks
----------------------------------------------------------
channel_general_chat.txt                                2
channel_igniters_information.txt                        8
channel_protocol_daily_updates.txt                      1
channel_solidroad_behavioral_assignments.txt            4
channel_staff_announcements.txt                         6
resource_additional_learning_material.txt               5
resource_community_map.txt                              2
resource_deliverables.txt                               1
resource_group_project_guidelines.txt                   3
resource_program_expectations.txt                       5
resource_solidroad.txt                                  4
resource_toolkit.txt                                    1
resource_welcome_page.txt                               2

--- First chunk preview ---
Source  : c

---
## Part 3 — Embeddings & ChromaDB Vector Store
**Future file:** `03_build_vectorstore.py`

Use `sentence-transformers/all-MiniLM-L6-v2` to embed chunks, then store in a local persistent ChromaDB collection.

- **Why this model?** Free, fast, runs locally, strong performance on semantic similarity tasks (~80MB).
- **ChromaDB** stores vectors in a local SQLite file — zero infra needed.
- Embeddings are **normalised** (cosine space), so retrieval uses cosine similarity.

> Run the *build* cell once. Subsequent runs will use the cached `load_vectorstore` path to avoid re-embedding.

In [7]:
# Load the embedding model (downloads ~80MB on first run)
print(f"Loading embedding model: {EMBEDDING_MODEL} ...")
embedding_model = SentenceTransformer(EMBEDDING_MODEL)
print("Embedding model loaded.")


def embed_texts(texts: list[str], batch_size: int = 64) -> list[list[float]]:
    """Embed a list of strings using the sentence-transformer model."""
    embeddings = embedding_model.encode(
        texts,
        batch_size=batch_size,
        normalize_embeddings=True,  # cosine space
        show_progress_bar=len(texts) > 50,
    )
    return embeddings.tolist()

Loading embedding model: sentence-transformers/all-MiniLM-L6-v2 ...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Embedding model loaded.


In [8]:
COLLECTION_NAME = "ask_andela"


def build_vectorstore(
    chunks: list[dict],
    persist_dir: Path,
    collection_name: str = COLLECTION_NAME,
    batch_size: int = 100,
) -> chromadb.Collection:
    """
    Embed all chunks and store them in a new ChromaDB collection.

    Drops and recreates the collection if it already exists.
    Use `load_vectorstore` to reuse an existing collection without re-embedding.
    """
    persist_dir.mkdir(parents=True, exist_ok=True)
    client = chromadb.PersistentClient(path=str(persist_dir))

    # Drop existing collection to rebuild from scratch
    existing = [c.name for c in client.list_collections()]
    if collection_name in existing:
        client.delete_collection(collection_name)
        print(f"Dropped existing collection '{collection_name}'")

    collection = client.create_collection(
        name=collection_name,
        metadata={"hnsw:space": "cosine"},
    )

    print(f"Embedding {len(chunks)} chunks in batches of {batch_size} ...")
    t0 = time.time()

    for i in range(0, len(chunks), batch_size):
        batch = chunks[i : i + batch_size]
        texts = [c["content"] for c in batch]
        metas = [c["metadata"] for c in batch]
        ids   = [f"chunk_{i + j}" for j in range(len(batch))]

        embeddings = embed_texts(texts)

        collection.add(
            documents=texts,
            embeddings=embeddings,
            metadatas=metas,
            ids=ids,
        )

        if (i // batch_size + 1) % 5 == 0 or (i + batch_size) >= len(chunks):
            done = min(i + batch_size, len(chunks))
            print(f"  Ingested {done}/{len(chunks)} chunks ...")

    elapsed = time.time() - t0
    print(f"\nVector store built in {elapsed:.1f}s")
    print(f"Collection '{collection_name}' has {collection.count()} vectors")
    return collection


def load_vectorstore(
    persist_dir: Path,
    collection_name: str = COLLECTION_NAME,
) -> chromadb.Collection:
    """Load an existing ChromaDB collection without re-embedding."""
    client = chromadb.PersistentClient(path=str(persist_dir))
    collection = client.get_collection(collection_name)
    print(f"Loaded collection '{collection_name}' with {collection.count()} vectors")
    return collection

In [9]:
# Build the vector store (re-run this cell to rebuild from scratch)
# To reuse an existing store, comment out build_vectorstore and uncomment load_vectorstore

collection = build_vectorstore(all_chunks, CHROMA_DIR)

# collection = load_vectorstore(CHROMA_DIR)  # ← use after the first build

Embedding 44 chunks in batches of 100 ...
  Ingested 44/44 chunks ...

Vector store built in 1.3s
Collection 'ask_andela' has 44 vectors


---
## Part 4 — Retrieval
**Future file:** `04_retrieval.py`

Given a user query, embed it and find the top-K most similar chunks in ChromaDB.
Returns chunks with their source document name, category, and relevance score.

In [10]:
def retrieve(
    query: str,
    collection: chromadb.Collection,
    top_k: int = TOP_K,
) -> list[dict]:
    """
    Embed the query and retrieve the top-K most relevant chunks.

    Returns a list of dicts:
        {
            "content":        str,
            "source":         str,   # filename (used as citation in UI)
            "category":       str,
            "chunk_index":    int,
            "relevance_score": float  # 0–1, higher is better
        }
    """
    query_embedding = embed_texts([query])

    results = collection.query(
        query_embeddings=query_embedding,
        n_results=top_k,
        include=["documents", "metadatas", "distances"],
    )

    retrieved = []
    for doc, meta, dist in zip(
        results["documents"][0],
        results["metadatas"][0],
        results["distances"][0],
    ):
        retrieved.append({
            "content":         doc,
            "source":          meta.get("source", "unknown"),
            "category":        meta.get("category", "unknown"),
            "chunk_index":     meta.get("chunk_index", 0),
            "relevance_score": round(1.0 - dist, 4),  # cosine similarity
        })

    return retrieved

In [11]:
# Smoke-test retrieval
test_query = "What is the capstone project and when is it due?"
retrieved_chunks = retrieve(test_query, collection, top_k=3)

print(f"Query: '{test_query}'\n")
for i, chunk in enumerate(retrieved_chunks, 1):
    print(f"[{i}] Source: {chunk['source']} | Score: {chunk['relevance_score']}")
    print(f"     Category: {chunk['category']}")
    print(f"     Content: {chunk['content'][:200].strip()}...")
    print()

Query: 'What is the capstone project and when is it due?'

[1] Source: resource_program_expectations.txt | Score: 0.4456
     Category: course_resource
     Content: Organizational Structure and Critical Skills for Success
- AI Project Roadmaps: Managing Uncertainty Through Strategic Planning
- AI Leadership: Fostering Adoption and Overcoming Implementation Barri...

[2] Source: resource_program_expectations.txt | Score: 0.3577
     Category: course_resource
     Content: 9    | Capstone   | Group project work             | Project 3            | Survey 3               |
| 10   | Capstone   | Assessment week                |                      | Showcase presentation...

[3] Source: resource_group_project_guidelines.txt | Score: 0.2622
     Category: course_resource
     Content: # LLM Engineering Post-Course Capstone — Group Project Guidelines
Source: Official course material, distributed to all participants in Week 3 of the A3: AI Engineering Bootcamp
Document type: Assessme...



---
## Part 5 — LLM via OpenRouter
**Future file:** `05_llm.py`

OpenRouter acts as a unified gateway to 200+ models (Llama, Qwen, Mistral, GPT-4o, Claude, etc.).
We use the standard OpenAI SDK client, pointing `base_url` at OpenRouter.

The system prompt instructs the model to stay grounded in course materials and respond in the cohort's teaching style.

In [12]:
# OpenRouter client — drop-in replacement for the OpenAI client
openrouter_client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=OPENROUTER_API_KEY,
)

SYSTEM_PROMPT = """You are Ask Andela, an AI study assistant for the A3 AI Engineering Bootcamp cohort.

You answer student questions using only the course materials and channel discussions provided as context.
Your answers should reflect the teaching style, tools, and terminology specific to this cohort — not generic internet knowledge.

Guidelines:
- Be concise: 3–6 sentences maximum per answer.
- Ground every answer in the provided context. If the context does not contain enough information, say so clearly.
- Never start with "As an AI language model..." or similar preambles.
- Reference specific course tools (ChromaDB, Gradio, QLoRA, OpenRouter, etc.) by name when relevant.
- Every answer must be self-contained — do not reference "the slide above" or "as mentioned earlier"."""


def build_rag_prompt(query: str, context_chunks: list[dict]) -> str:
    """Assemble the user-turn message with retrieved context."""
    context_blocks = []
    for chunk in context_chunks:
        context_blocks.append(
            f"[Source: {chunk['source']} | Category: {chunk['category']}]\n{chunk['content']}"
        )
    context_text = "\n\n---\n\n".join(context_blocks)

    return f"""Use the following course materials to answer the student's question.
If the context does not cover the question, say so honestly.

CONTEXT:
{context_text}

STUDENT QUESTION:
{query}"""


def generate_answer(
    query: str,
    context_chunks: list[dict],
    model: str = LLM_MODEL,
    temperature: float = 0.1,
    max_tokens: int = 512,
) -> str:
    """Call the OpenRouter LLM and return the generated answer."""
    user_prompt = build_rag_prompt(query, context_chunks)

    response = openrouter_client.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user",   "content": user_prompt},
        ],
        temperature=temperature,
        max_tokens=max_tokens,
        extra_headers={
            "HTTP-Referer": "https://github.com/ask-andela",
            "X-Title": "Ask Andela",
        },
    )
    return response.choices[0].message.content.strip()

In [ ]:
# Smoke-test the LLM with a hardcoded context snippet
dummy_chunks = [{
    "content": "The capstone is a 2-day group project. Deliverable: a working local Gradio demo. No production deployment required.",
    "source": "resource_group_project_guidelines.txt",
    "category": "course_resource",
    "chunk_index": 0,
}]

test_answer = generate_answer(
    "What do we need to submit for the capstone?",
    dummy_chunks,
    model=LLM_MODEL,
)
print("LLM test answer:")
print(test_answer)

LLM test answer:
For the capstone, you need to submit a working local Gradio demo. This is the primary deliverable.


---
## Part 6 — Full RAG Pipeline
**Future file:** `06_rag_pipeline.py`

The complete end-to-end pipeline:
1. Embed user query
2. Retrieve top-K chunks from ChromaDB
3. Assemble prompt with context
4. Call LLM via OpenRouter
5. Return answer + deduplicated source citations

In [18]:
def ask_andela(
    query: str,
    collection: chromadb.Collection,
    top_k: int = TOP_K,
    model: str = LLM_MODEL,
) -> dict:
    """
    Full RAG pipeline: query → retrieve → generate → cite.

    Returns:
        {
            "answer":         str,
            "sources":        list[str],   # deduplicated source filenames
            "context_chunks": list[dict],  # raw retrieved chunks (for debugging)
        }
    """
    # Step 1: Retrieve relevant context
    context_chunks = retrieve(query, collection, top_k)

    # Step 2: Generate answer with LLM
    answer = generate_answer(query, context_chunks, model)

    # Step 3: Deduplicate + order sources by relevance score
    seen = set()
    sources = []
    for chunk in sorted(context_chunks, key=lambda c: c["relevance_score"], reverse=True):
        if chunk["source"] not in seen:
            sources.append(chunk["source"])
            seen.add(chunk["source"])

    return {
        "answer":         answer,
        "sources":        sources,
        "context_chunks": context_chunks,
    }

In [22]:
# Test the full RAG pipeline with a few questions
test_questions = [
    "What is the group project and when is it due?",
    "How do I set up the environment for fine-tuning on Colab?",
    "What tools are we using for the RAG pipeline?",
    "How many students from Igniters completed the week 6 PRs?",
    "How can I debug the 'openai module not found' error in VS Code?"
]

for question in test_questions:
    print(f"Q: {question}")
    print("-" * 60)
    result = ask_andela(question, collection)
    print(f"A: {result['answer']}")
    print(f"\nSources: {', '.join(result['sources'])}")
    print("=" * 70)
    print()

Q: What is the group project and when is it due?
------------------------------------------------------------
A: The group project is a 3-Day Group Project where each group of 4-5 students will design and build a real-world AI application over 2 days. The project is due after the 2-day implementation period, with a final presentation on Friday.

Sources: resource_program_expectations.txt, channel_staff_announcements.txt, resource_group_project_guidelines.txt

Q: How do I set up the environment for fine-tuning on Colab?
------------------------------------------------------------
A: Unfortunately, the provided context does not contain information on setting up the environment for fine-tuning on Colab. The context mentions fine-tuning as one of the focus areas for the group project in Week 3, but it does not provide instructions on how to set it up.

Sources: channel_solidroad_behavioral_assignments.txt, resource_program_expectations.txt, resource_group_project_guidelines.txt, resource_c

---
## Part 7 — Gradio UI
**Future file:** `07_gradio_ui.py`

A simple, clean Gradio interface:
- **Input:** Student question (text box)
- **Output:** AI answer + formatted source citations

The UI is intentionally minimal for the demo — matches the PRD's single-turn Q&A scope.

In [23]:
def gradio_query(question: str) -> tuple[str, str]:
    """
    Gradio-compatible wrapper. Returns (answer, sources_text).
    Handles empty input and API errors gracefully.
    """
    if not question or not question.strip():
        return "Please enter a question.", ""

    try:
        result = ask_andela(question.strip(), collection)
        sources_text = "**Sources cited:**\n" + "\n".join(
            f"• {src}" for src in result["sources"]
        )
        return result["answer"], sources_text
    except Exception as e:
        return f"Error: {str(e)}", ""


# Example questions shown in the UI
EXAMPLE_QUESTIONS = [
    "What is RAG and why do we use it in this course?",
    "How do I install VS Build Tools on Windows?",
    "What is QLoRA and why is it used for fine-tuning?",
    "What are the submission requirements for the capstone project?",
    "How does ChromaDB store embeddings?",
    "What model should I use for fine-tuning on Google Colab?",
    "When are behavioral assignments due?",
    "What is the format of submitting daily squad updates?"
]

with gr.Blocks(
    title="Ask Andela",
    theme=gr.themes.Soft(primary_hue="blue"),
    css="""
    .gradio-container { max-width: 860px; margin: auto; }
    .source-box { font-size: 0.85em; color: #555; }
    """,
) as demo:

    gr.Markdown(
        """
        # 🎓 Ask Andela
        **Your AI study assistant for the A3 AI Engineering Bootcamp.**
        Answers are grounded in the cohort's own course materials and channel discussions.
        """
    )

    with gr.Row():
        with gr.Column(scale=3):
            question_box = gr.Textbox(
                label="Your question",
                placeholder="e.g. What is the difference between RAG and fine-tuning?",
                lines=2,
            )
            submit_btn = gr.Button("Ask", variant="primary")

    with gr.Row():
        with gr.Column():
            answer_box = gr.Textbox(
                label="Answer",
                lines=6,
                interactive=False,
            )
            sources_box = gr.Markdown(
                label="Sources",
                elem_classes=["source-box"],
            )

    gr.Examples(
        examples=[[q] for q in EXAMPLE_QUESTIONS],
        inputs=question_box,
        label="Try these examples",
    )

    submit_btn.click(
        fn=gradio_query,
        inputs=question_box,
        outputs=[answer_box, sources_box],
    )
    question_box.submit(
        fn=gradio_query,
        inputs=question_box,
        outputs=[answer_box, sources_box],
    )

    gr.Markdown(
        """
        ---
        *Powered by [OpenRouter](https://openrouter.ai) · Embeddings: all-MiniLM-L6-v2 · Vector store: ChromaDB*
        """
    )

# Launch the app
# Set share=True to get a public Gradio link (useful for demo)
demo.launch(share=False)

/var/folders/n2/vpj9gn8j5n34yqpdc6r5frhw0000gn/T/ipykernel_11658/3502784990.py:31: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(
/var/folders/n2/vpj9gn8j5n34yqpdc6r5frhw0000gn/T/ipykernel_11658/3502784990.py:31: DeprecationWarning: The 'css' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'css' to Blocks.launch() instead.
  with gr.Blocks(


* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


---
## Part 8 — Evaluation Framework
**Future file:** `08_evaluation.py`

Per the PRD, we evaluate on 10–15 held-out questions comparing:
1. **Base model** (no RAG, no fine-tuning) — direct LLM call
2. **RAG pipeline** (no fine-tuning) — current notebook
3. **Fine-tuned model + RAG** — to be wired in after QLoRA training

Each response is scored 1–3 on **Accuracy**, **Specificity**, and **Conciseness**.

This section sets up the eval harness and saves results to `eval_results.json`.

In [ ]:
# Held-out evaluation questions (NOT used in fine-tuning dataset)
EVAL_QUESTIONS = [
    # Technical — RAG & vector stores
    {"id": "eval_01", "question": "What is RAG and why is it used in this course instead of just prompting the LLM directly?"},
    {"id": "eval_02", "question": "How does ChromaDB store and retrieve embeddings?"},
    {"id": "eval_03", "question": "What embedding model do we use and why was it chosen?"},
    # Technical — fine-tuning
    {"id": "eval_04", "question": "What is QLoRA and why do we use it instead of full fine-tuning?"},
    {"id": "eval_05", "question": "Which base model should I fine-tune on Google Colab and why?"},
    # Technical — tools
    {"id": "eval_06", "question": "How do I set up the fine-tuning environment on Colab?"},
    {"id": "eval_07", "question": "What is the role of Gradio in the final demo?"},
    # Logistics — programme
    {"id": "eval_08", "question": "What are the deliverables for the capstone project and how is it assessed?"},
    {"id": "eval_09", "question": "When are behavioral assignments due each week?"},
    # Setup / troubleshooting
    {"id": "eval_10", "question": "How do I fix the 'openai module not found' error in VS Code?"},
    {"id": "eval_11", "question": "How do I install VS Build Tools on Windows?"},
    # Programme norms
    {"id": "eval_12", "question": "What is the difference between peer review and the capstone assessment?"},
]

print(f"Evaluation set: {len(EVAL_QUESTIONS)} questions")

In [ ]:
def baseline_answer(question: str, model: str = LLM_MODEL) -> str:
    """
    Call the LLM with NO retrieval context — baseline comparison.
    Uses a neutral system prompt (no cohort-specific grounding).
    """
    response = openrouter_client.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": "You are a helpful AI assistant. Answer the following question concisely."},
            {"role": "user",   "content": question},
        ],
        temperature=0.1,
        max_tokens=512,
    )
    return response.choices[0].message.content.strip()


def run_evaluation(
    eval_questions: list[dict],
    collection: chromadb.Collection,
    output_path: Optional[Path] = None,
    run_baseline: bool = True,
) -> list[dict]:
    """
    Run evaluation comparing baseline vs RAG for each question.

    Results are saved to `output_path` if provided.
    Fine-tuned model column is left as a placeholder for Kayode's integration step.
    """
    results = []

    for i, item in enumerate(eval_questions, 1):
        q_id = item["id"]
        question = item["question"]
        print(f"[{i}/{len(eval_questions)}] {q_id}: {question[:60]}...")

        result_entry = {
            "id":       q_id,
            "question": question,
            "baseline_answer":    None,
            "rag_answer":         None,
            "rag_sources":        None,
            "finetuned_rag_answer": None,  # placeholder for Part 2 (fine-tuning integration)
            # Scoring columns (to be filled in manually during demo)
            "baseline_accuracy":    None,
            "baseline_specificity": None,
            "baseline_conciseness": None,
            "rag_accuracy":         None,
            "rag_specificity":      None,
            "rag_conciseness":      None,
            "finetuned_rag_accuracy":    None,
            "finetuned_rag_specificity": None,
            "finetuned_rag_conciseness": None,
        }

        # Baseline (no RAG)
        if run_baseline:
            result_entry["baseline_answer"] = baseline_answer(question)

        # RAG pipeline
        rag_result = ask_andela(question, collection)
        result_entry["rag_answer"]   = rag_result["answer"]
        result_entry["rag_sources"]  = rag_result["sources"]

        results.append(result_entry)

    if output_path:
        output_path.parent.mkdir(parents=True, exist_ok=True)
        with open(output_path, "w", encoding="utf-8") as f:
            json.dump(results, f, indent=2, ensure_ascii=False)
        print(f"\nResults saved to {output_path}")

    return results

In [ ]:
# Run the evaluation (this will make ~24 API calls — takes ~2–3 minutes)
# Set run_baseline=False to skip baseline calls and save API spend

EVAL_OUTPUT = PROJECT_ROOT / "eval_results.json"

eval_results = run_evaluation(
    EVAL_QUESTIONS,
    collection,
    output_path=EVAL_OUTPUT,
    run_baseline=True,
)

In [ ]:
# Display evaluation results as a comparison table
print(f"{'ID':<10} {'Question (truncated)':<45} {'Baseline (first 80 chars)':<82} {'RAG answer (first 80 chars)'}")
print("-" * 230)

for r in eval_results:
    baseline_preview = (r["baseline_answer"] or "")[:80].replace("\n", " ")
    rag_preview      = (r["rag_answer"]      or "")[:80].replace("\n", " ")
    q_preview        = r["question"][:43]
    print(f"{r['id']:<10} {q_preview:<45} {baseline_preview:<82} {rag_preview}")

print(f"\nFull results saved to: {EVAL_OUTPUT}")
print("\nNOTE: Manually score each response 1–3 on Accuracy, Specificity, Conciseness.")
print("      Then re-run after fine-tuning integration to fill the 'finetuned_rag_answer' column.")

---
## Next Steps — Fine-Tuning Integration

The cells below are stubs for **Day 2** integration with the QLoRA fine-tuned model.
Kayode will wire the fine-tuned adapter here after Colab training completes.

The idea:
1. Load the base model + LoRA adapter from HuggingFace Hub (or local path)
2. Replace `generate_answer()` with an `ask_andela_finetuned()` variant
3. Re-run the evaluation with the `finetuned_rag_answer` column populated
4. Compare the three columns in the demo table

In [ ]:
# ── Fine-tuned model integration stub ────────────────────────────────────────
# TODO (Kayode): fill in after QLoRA training on Colab
#
# from transformers import AutoTokenizer, AutoModelForCausalLM
# from peft import PeftModel
#
# BASE_MODEL_ID    = "Qwen/Qwen2.5-0.5B-Instruct"   # or "microsoft/Phi-3-mini-4k-instruct"
# ADAPTER_PATH     = "path/to/lora_adapter"          # local path or HF repo id
#
# tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_ID)
# base_model = AutoModelForCausalLM.from_pretrained(BASE_MODEL_ID, torch_dtype="auto", device_map="auto")
# finetuned_model = PeftModel.from_pretrained(base_model, ADAPTER_PATH)
#
# def generate_finetuned_answer(query: str, context_chunks: list[dict]) -> str:
#     prompt = build_rag_prompt(query, context_chunks)
#     inputs = tokenizer(prompt, return_tensors="pt").to(finetuned_model.device)
#     with torch.no_grad():
#         output = finetuned_model.generate(**inputs, max_new_tokens=256, temperature=0.1)
#     return tokenizer.decode(output[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)

print("Fine-tuning stub ready. Fill in ADAPTER_PATH after Colab training.")